# 保留文档修订与去重台账

## 目标

输出截止时点内的全部已观察版本，并区分首次观察、内容变更与内容未变。不识别情绪，也不自动判断新闻真伪。

本文件使用虚构教学数据，不是论文复现或生产数据。

## 准备

使用 Python 3.10+ 内核，按顺序运行全部单元格。计算仅依赖标准库，无需密钥、联网或额外数据文件。可在已有的 Jupyter 环境中打开。

输入已内嵌，与同目录 inputs.json 内容一致；可在下一个单元格中修改 args 试验。时间与单位必须显式保留。

In [ ]:
import json

# Synthetic inputs; no credentials or network access.
bundle = json.loads("{\"version\":1,\"tutorial\":\"document-version-ledger\",\"identity\":\"synthetic\",\"args\":[[{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v1\",\"contentHash\":\"aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa\",\"publishedAt\":\"2025-01-06T08:00:00Z\",\"firstSeenAt\":\"2025-01-06T08:01:00Z\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v1\",\"contentHash\":\"aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa\",\"publishedAt\":\"2025-01-06T08:00:00Z\",\"firstSeenAt\":\"2025-01-06T08:01:00Z\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v2\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-07T08:00:00Z\",\"firstSeenAt\":\"2025-01-07T08:01:00Z\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v3\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-08T08:00:00Z\",\"firstSeenAt\":\"2025-01-08T08:01:00Z\"}],\"2025-01-08T09:00:00Z\"],\"expected\":[{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v1\",\"contentHash\":\"aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa\",\"publishedAt\":\"2025-01-06T08:00:00Z\",\"firstSeenAt\":\"2025-01-06T08:01:00Z\",\"availableAt\":\"2025-01-06T08:01:00.000Z\",\"status\":\"first_observation\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v2\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-07T08:00:00Z\",\"firstSeenAt\":\"2025-01-07T08:01:00Z\",\"availableAt\":\"2025-01-07T08:01:00.000Z\",\"status\":\"changed_content\"},{\"publisher\":\"DEMO\",\"documentId\":\"DOC-1\",\"version\":\"v3\",\"contentHash\":\"bbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbbb\",\"publishedAt\":\"2025-01-08T08:00:00Z\",\"firstSeenAt\":\"2025-01-08T08:01:00Z\",\"availableAt\":\"2025-01-08T08:01:00.000Z\",\"status\":\"unchanged_content\"}]}")
args = bundle["args"]
expected = bundle["expected"]
print(json.dumps(args, ensure_ascii=False, indent=2))

## 步骤

### 1. 先定义文档与版本

文档身份应在发布方范围内稳定，修订号也必须可追溯。标题相同不证明是同一文件；不同发布方即使内容一样，也保留各自身份。示例不做转载聚类或实体识别，真实公告中的URL、公司代码与原始文件还需要作为独立来源证据保存。

### 2. 冻结内容哈希口径

实际哈希必须来自固定口径的原始字节或规范化文本，并记录算法与解析版本。两者不能混用，否则PDF元数据变化或解析器更新也可能看似内容修订。示例里的64位字符串只是虚构标识，函数不下载文件、不计算真实哈希，也不替代来源完整性验证。

### 3. 先查冲突，再限定可得性

完全相同的版本重复输入只保留一次；同一版本出现不同哈希或时间证据则停止。按发布时间与首次观察时间的较晚者筛选截止时点，不能把后来抓到的文件倒填成过去已持有。只有日期的来源不能直接补零点，需要外部核对或保守规则。

### 4. 保留每次变化的轨迹

对每份文档按可得时间排列版本，并与上一次可得版本比较哈希：标记首次观察、内容变化或内容未变。相同可得时间的不同版本拒绝猜顺序。内容变化不表示哪版更真实，也不自动代表负面消息；需要回到原文理解具体修订。

### 方法与假设

- 版本台账需要自己保留的历史证据，最新API结果无法自动补出它。
- 相同哈希不等于相同版权、发布方或文件身份。
- 真实许可与全文获取范围需单独核对。

In [ ]:
from datetime import datetime, timezone
import json
import re


def preserve_document_versions(rows, cutoff):
    def parse(value):
        try:
            parsed = datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            if parsed.strftime("%Y-%m-%dT%H:%M:%SZ") != value:
                raise ValueError()
            return parsed
        except (ValueError, TypeError):
            raise ValueError("utc_seconds_required")

    boundary, identities, eligible = parse(cutoff), {}, []
    for row in rows:
        if not all(isinstance(row.get(key), str) and row[key] for key in ("publisher", "documentId", "version")) or not isinstance(row.get("contentHash"), str) or not re.fullmatch(r"[a-f0-9]{64}", row["contentHash"]):
            raise ValueError("invalid_document_identity")
        available = max(parse(row.get("publishedAt")), parse(row.get("firstSeenAt")))
        key = (row["publisher"], row["documentId"], row["version"])
        fingerprint = (row["contentHash"], row["publishedAt"], row["firstSeenAt"])
        if key in identities:
            if identities[key] != fingerprint:
                raise ValueError("conflicting_document_version")
            continue
        identities[key] = fingerprint
        if available <= boundary:
            eligible.append({**row, "availableAt": available.isoformat(timespec="milliseconds").replace("+00:00", "Z")})
    eligible.sort(key=lambda row: (row["availableAt"], json.dumps([row["publisher"], row["documentId"], row["version"]], ensure_ascii=False, separators=(",", ":"))))
    previous, result = {}, []
    for row in eligible:
        key = (row["publisher"], row["documentId"])
        prior = previous.get(key)
        if prior and prior["availableAt"] == row["availableAt"]:
            raise ValueError("ambiguous_revision_order")
        previous[key] = row
        status = "first_observation" if not prior else "unchanged_content" if prior["contentHash"] == row["contentHash"] else "changed_content"
        result.append({**row, "status": status})
    return result


### 运行小样本

4条输入去重后留下3个版本：v1首次观察，v2内容变化，v3内容未变；不覆盖旧版本。

In [ ]:
result = preserve_document_versions(*args)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 检查

将每一行与网页示例的预期输出比较。修改输入后，断言失败可能正是预期结果：先解释差异，不要直接删除验证。

In [ ]:
assert result == expected, "Output differs from the reference synthetic example"
assert bundle["identity"] == "synthetic"
print("通过：结果与网页虚构示例一致。")

## 下一步

真实数据须先通过已认证 GET /v1/catalog 核对权限、字段、schema_major、窗口与来源，再按实际合同映射。这里列出的是候选输入身份，不保证可用或历史完整。不要把 API as_of 当作历史财报版本。真实输入替换后须重新验证；不要沿用这份小样本的通过结论。

- `cn.dataset.anns_d`
- `cn.news.flash`

### 参考资料

- [Tushare：公告字段与原文链接](https://tushare.pro/document/2?doc_id=176)
- [Loughran与McDonald：解析范围](https://www.uts.edu.au/globalassets/sites/default/files/adg_cons2015_loughran-mcdonald-je-2011.pdf)

[返回教程](https://tradingdatas.com/recipes/document-version-ledger/)